In [1]:
#imports
import pandas as pd
from pathlib import Path
import plotly.express as px

In [2]:
#LOAD AVAILABLE DATA

folder = Path(r"C:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\results")

csv_dataframes = {}

for csv_file in folder.glob("*.csv"):
    df_name = csv_file.stem
    csv_dataframes[df_name] = pd.read_csv(csv_file)

print(f"Loaded {len(csv_dataframes)} CSV files:")
for name, df in csv_dataframes.items():
    print(f"{name}: {df.shape}")

# Add method name from CSV filename
all_data = []

for method_name, df in csv_dataframes.items():
    temp = df.copy()
    temp["method"] = method_name
    all_data.append(temp)

long_df = pd.concat(all_data, ignore_index=True)

long_df.head()


id_col = "subject name"      # change if needed
method_col = "method"   # change if needed

metric_columns = [
    col for col in long_df.select_dtypes(include="number").columns
    if col not in [id_col]
]

method_options = sorted(long_df[method_col].unique())

Loaded 15 CSV files:
Dense_raycast: (10, 11)
nnUnet: (10, 11)
Recontour_B_results: (10, 11)
Recontour_C_results: (10, 11)
Recontour_D_results: (10, 11)
Recontour_E_results: (10, 11)
Sparse_normals: (10, 11)
Sparse_normals_partitionpropagation_1: (10, 11)
Sparse_normals_partitionpropagation_5: (10, 11)
Sparse_normals_scaledplacement: (10, 11)
Sparse_raycast: (10, 11)
weighted_multiprop_optimtest: (10, 11)
weighted_multiprop_promptsets_results: (10, 11)
weighted_multiprop_promptsets_results_0_0_0.1_0.9: (10, 11)
weighted_multiprop_promptsets_results_0_0_0.2_0.8: (10, 11)


In [3]:
# INTERACTIVE DESCRIPTIVE STATISTICS FOR ALL METRICS

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

def compute_descriptive_statistics(selected_metrics, selected_methods, round_decimals=3):

    selected_metrics = list(selected_metrics)
    selected_methods = list(selected_methods)

    df = long_df[long_df[method_col].isin(selected_methods)].copy()

    all_stats = []

    for metric in selected_metrics:

        paired_df = df.pivot(
            index=id_col,
            columns=method_col,
            values=metric
        ).dropna()

        if paired_df.empty:
            continue

        stats_df = paired_df.describe().T

        stats_df["median"] = paired_df.median()
        stats_df["iqr"] = paired_df.quantile(0.75) - paired_df.quantile(0.25)
        stats_df["missing_values"] = paired_df.isna().sum()
        stats_df["metric"] = metric

        all_stats.append(stats_df)

    if len(all_stats) == 0:
        print("No valid paired data found.")
        return

    final_stats = pd.concat(all_stats)

    final_stats = final_stats.reset_index().rename(
        columns={"index": "method"}
    )

    final_stats = final_stats[
        [
            "metric",
            "method",
            "count",
            "mean",
            "std",
            "min",
            "25%",
            "median",
            "75%",
            "iqr",
            "max",
            "missing_values"
        ]
    ]

    display(final_stats.round(round_decimals))


# Automatically get numeric metric columns
available_metrics = [
    col for col in long_df.select_dtypes(include="number").columns
    if col != id_col
]

metric_selector = widgets.SelectMultiple(
    options=available_metrics,
    value=tuple(available_metrics),
    description="Metrics:",
    rows=min(10, len(available_metrics))
)

method_selector = widgets.SelectMultiple(
    options=method_options,
    value=tuple(method_options),
    description="Methods:",
    rows=min(8, len(method_options))
)

round_slider = widgets.IntSlider(
    value=3,
    min=0,
    max=6,
    step=1,
    description="Decimals:"
)

out = widgets.interactive_output(
    compute_descriptive_statistics,
    {
        "selected_metrics": metric_selector,
        "selected_methods": method_selector,
        "round_decimals": round_slider
    }
)

display(
    widgets.VBox([
        metric_selector,
        method_selector,
        round_slider
    ]),
    out
)

Output()

In [4]:
# INTERACTIVE PLOTLY DASHBOARD
def analyze_metric_plotly(
    metric,
    selected_methods,
    show_boxplot,
    show_paired_lines,
):

    selected_methods = list(selected_methods)

    if len(selected_methods) < 2:
        print("Select at least two methods.")
        return

    # Filter methods
    df = long_df[long_df[method_col].isin(selected_methods)].copy()

    # Create paired dataframe
    paired_df = df.pivot(
        index=id_col,
        columns=method_col,
        values=metric
    )

    paired_df = paired_df.dropna()

    print(f"\nMetric: {metric}")
    print(f"Number of paired samples: {len(paired_df)}")

    # Reset subject numbering
    paired_df = paired_df.reset_index(drop=True)

    # ---------------- BOXPLOT ----------------

    if show_boxplot:

        box_df = paired_df.melt(
            var_name="Method",
            value_name=metric
        )

        fig = px.box(
            box_df,
            x="Method",
            y=metric,
            points="all",
            title=f"Paired comparison of {metric}"
        )

        fig.update_layout(
            height=500,
            width=900
        )

        fig.show()

    # ---------------- PAIRED LINE PLOT ----------------

    if show_paired_lines:

        fig = px.line(
            paired_df,
            markers=True,
            labels={
                "index": "Subject Number",
                "value": metric,
                "variable": "Method"
            },
            title=f"{metric} per subject"
        )

        fig.update_layout(
            xaxis_title="Subject Number",
            yaxis_title=metric,
            height=600,
            width=1000
        )

        fig.show()


# ---------------- WIDGETS ----------------

metric_dropdown = widgets.Dropdown(
    options=metric_columns,
    description="Metric:"
)

methods_select = widgets.SelectMultiple(
    options=method_options,
    value=tuple(method_options),
    description="Methods:"
)

boxplot_checkbox = widgets.Checkbox(
    value=True,
    description="Boxplot"
)

paired_checkbox = widgets.Checkbox(
    value=True,
    description="Paired lines"
)


ui = widgets.VBox([
    metric_dropdown,
    methods_select,
    boxplot_checkbox,
    paired_checkbox,
])

out = widgets.interactive_output(
    analyze_metric_plotly,
    {
        "metric": metric_dropdown,
        "selected_methods": methods_select,
        "show_boxplot": boxplot_checkbox,
        "show_paired_lines": paired_checkbox,
    }
)

display(ui, out)

Output()